In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
BASE_PATH = '/kaggle/input/llm-classification-finetuning'
TRAIN_DATA = BASE_PATH + '/train.csv'
TEST_DATA = BASE_PATH + '/test.csv'

## Config

In [ ]:
import tensorflow as tf

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("✅ TPU ready:", tpu.master())
except Exception as e:
    print("⚠️ GPU/CPU fallback:", e)
    strategy = tf.distribute.get_strategy()
print("🔧 # replicas:", strategy.num_replicas_in_sync)

In [ ]:
ENABLE_SUBMISSION = True
MODEL_NAME = "deberta_v3_small_en"
SEQUENCE_LENGTH = 128
BATCH_SIZE = 8
EPOCH = 5
CLASSES = 3
ENCODE_TYPE = "utf-8"
DROPOUT = 0.5
SHUFFLE_SEED = 42
SHUFFLE_BUFFER_SIZE = 1024
WEIGHT_DECAY = 0.01
LOSS_LABEL_SMOOTHING = 0.1
LEARNING_RATE = 5e-6
SPLIT_DS = 0.1

## Load dataset

In [ ]:
import pandas as pd
df = pd.read_csv(TRAIN_DATA)
test_df = pd.read_csv(TEST_DATA)

### Clean train df

In [ ]:
def make_label(row):
    if row['winner_model_a'] == 1:
        return 0
    elif row['winner_model_b'] == 1:
        return 1
    else:
        return 2

import ast
def extract_text(text):
    try:
        return ast.literal_eval(text)[0]
    except Exception:
        return text  # fallback if something goes wrongreturn text[2:-2]

# Make label
df["class_name"] = df[["winner_model_a", "winner_model_b" , "winner_tie"]].idxmax(axis=1)
df["label"] = df.apply(make_label, axis=1)

# Extract text
df["prompt"] = df['prompt'].apply(extract_text)
df["response_a"] = df['response_a'].apply(extract_text)
df["response_b"] = df['response_b'].apply(extract_text)

df.head()

### See data distribution

In [ ]:
# distribution of LLMs in df
model_df = pd.concat([df.model_a, df.model_b])
model_count = model_df.value_counts()
model_count = model_count.reset_index()
model_count.columns = ['LLM', 'count']
model_count.head()

In [ ]:
# How many records for each label?
df['label'].value_counts()
# total = 20064 + 19652 + 17761 = 57477

# percent_0 = 20064 / 57477 = 34.9%
# percent_1 = 19652 / 57477 = 34.2%
# percent_2 = 17761 / 57477 = 30.9%

### Clean test_df

In [ ]:
# Extract text
test_df["prompt"] = test_df['prompt'].apply(extract_text)
test_df["response_a"] = test_df['response_a'].apply(extract_text)
test_df["response_b"] = test_df['response_b'].apply(extract_text)

test_df.head()

### Create column (Prompt + Response A) and (Prompt + Response B)

In [ ]:
import re
import string

def clean_text(text): 
    text = re.sub(r"\s+", " ", text)         # Collapse multiple spaces/newlines
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    return text.strip()


df["prompt"] = df['prompt'].apply(clean_text)
df["response_a"] = df['response_a'].apply(clean_text)
df["response_b"] = df['response_b'].apply(clean_text)

test_df["prompt"] = test_df['prompt'].apply(clean_text)
test_df["response_a"] = test_df['response_a'].apply(clean_text)
test_df["response_b"] = test_df['response_b'].apply(clean_text)

In [ ]:
import keras_nlp

# Load the tokenizer from preset
tokenizer = keras_nlp.models.DebertaV3Tokenizer.from_preset(MODEL_NAME)

def safe_encode(text):
    try:
        return text.encode(ENCODE_TYPE).decode(ENCODE_TYPE), False
    except:
        return "", True

def build_inputs(row):
    row["encode_fail"] = False

    prompt, fail_prompt = safe_encode(row.get("prompt", ""))
    response_a, fail_a = safe_encode(row.get("response_a", ""))
    response_b, fail_b = safe_encode(row.get("response_b", ""))

    # If any encoding failed, mark as failed
    row["encode_fail"] = fail_prompt or fail_a or fail_b

    row["inputs"] = [
        f"Prompt: {prompt} {tokenizer.sep_token} Response: {response_a}",
        f"Prompt: {prompt} {tokenizer.sep_token} Response: {response_b}"
    ]

    return row
        
# apply to train_df
df = df.apply(build_inputs, axis=1)

# apply to test_df
test_df = test_df.apply(build_inputs, axis=1)

In [ ]:
df[["inputs", "label"]].head()

In [ ]:
df['inputs'][0]

In [ ]:
# check how many rows are encoded fail
display(df['encode_fail'].value_counts())
display(test_df['encode_fail'].value_counts())

In [ ]:
# Splitting Dataset
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(df, test_size=SPLIT_DS, stratify=df["label"])

In [ ]:
train_df

## Create dataloader

In [ ]:
import keras
import keras_hub
import numpy as np
import tensorflow as tf

preprocessor = keras_hub.models.DebertaV3Preprocessor.from_preset(
    MODEL_NAME,
    sequence_length=SEQUENCE_LENGTH,
)

def preprocess_fn(text, label=None):
    text = preprocessor(text)  # Preprocess text
    return (text, label) if label is not None else text


# Dataset
def build_dataset(texts, labels=None, batch_size=BATCH_SIZE, shuffle=True, num_classes=CLASSES):
    AUTO = tf.data.AUTOTUNE

    # If labels are provided, convert to categorical (one-hot)
    if labels is not None:
        labels = tf.keras.utils.to_categorical(labels, num_classes=num_classes)
        slices = (texts, labels)
    else:
        slices = (texts,)

    # Create the tf.data.Dataset from slices
    ds = tf.data.Dataset.from_tensor_slices(slices)

    # Shuffle if requested (only if labels exist — e.g., training/val)
    if shuffle and labels is not None:
        ds = ds.shuffle(buffer_size=SHUFFLE_BUFFER_SIZE, seed=SHUFFLE_SEED, reshuffle_each_iteration=True)

    # Preprocess each item
    ds = ds.map(preprocess_fn, num_parallel_calls=AUTO)

    # Batch and prefetch
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(AUTO)

    return ds

# Dataloader
train_texts = train_df['inputs'].tolist()
train_labels = train_df['label'].tolist()
train_ds = build_dataset(train_texts, train_labels,
                         batch_size=BATCH_SIZE,
                         shuffle=True)

# validation set dataloader
valid_texts = valid_df['inputs'].tolist()
valid_labels = valid_df['label'].tolist()
valid_ds = build_dataset(valid_texts, valid_labels,
                         batch_size=BATCH_SIZE,
                         shuffle=False)

# test set dataloader
test_texts = test_df["inputs"].tolist()
test_ds = build_dataset(test_texts, labels=None, batch_size=min(len(test_texts), BATCH_SIZE), 
                        shuffle=False)

In [ ]:
for batch in train_ds.take(1):
    x, y = batch  # x = inputs, y = labels

    print("Input keys:", x.keys())
    print("token_ids shape:", x["token_ids"].shape)
    print("padding_mask shape:", x["padding_mask"].shape)
    print("Labels shape:", y.shape)
    print("First few labels:", y[:5])

## Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Define the callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,                # Stop if val_loss doesn't improve for 2 epochs
    restore_best_weights=True # Restore weights from the best epoch
)

lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,        # Reduce LR by half
    patience=1,        # Wait 1 epoch with no improvement
    min_lr=1e-6        # Never go below this learning rate
)

checkpoint = keras.callbacks.ModelCheckpoint(
    "best_model.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    mode='min' 
)

In [ ]:
import tensorflow as tf
from tensorflow import keras
import keras_nlp

class PreferenceClassifier(keras.Model):
    def __init__(self, preset, num_classes=CLASSES):
        super().__init__()
        self.present = preset

        # Backbone encoder
        self.backbone = keras_nlp.models.DebertaV3Backbone.from_preset(preset)

        # Pooling + classification head
        self.pooling = keras.layers.GlobalAveragePooling1D()
        self.dropout = keras.layers.Dropout(DROPOUT)
        self.classifier = keras.layers.Dense(num_classes, activation="softmax")

    def call(self, inputs, training=False):
        """
        inputs: {
            "token_ids": Tensor of shape (batch_size, 2, seq_len),
            "padding_mask": Tensor of shape (batch_size, 2, seq_len)
        }
        """
        # Slice inputs for response A and B
        input_a = {k: v[:, 0, :] for k, v in inputs.items()}
        input_b = {k: v[:, 1, :] for k, v in inputs.items()}
    
        # Get contextual embeddings from backbone
        embed_a = self.backbone(input_a, training=training)
        embed_b = self.backbone(input_b, training=training)
    
        # Combine embeddings
        combined = tf.concat([embed_a, embed_b], axis=-1)
    
        # Pool and classify
        x = self.pooling(combined)
        x = self.dropout(x, training=training) 
        return self.classifier(x)

with strategy.scope():
    model = PreferenceClassifier(preset=MODEL_NAME, num_classes=CLASSES)
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LOSS_LABEL_SMOOTHING),
        metrics=["accuracy"]
    )

model.fit(train_ds, 
          validation_data=valid_ds, 
          epochs=EPOCH,
          callbacks=[early_stopping, checkpoint, lr_scheduler])

## Inference

In [ ]:
# Make predictions using the trained model on test data
model.load_weights('/kaggle/working/best_model.weights.h5')
test_preds = model.predict(test_ds, verbose=1)

In [ ]:
test_preds

In [ ]:
if ENABLE_SUBMISSION:
    test_ids = test_df['id']
    submission = pd.DataFrame({
            "id": test_ids,
            "winner_model_a": test_preds[:, 0],
            "winner_model_b": test_preds[:, 1],
            "winner_tie": test_preds[:, 2]
        })
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print("✅ Submission file saved: submission.csv")